# CSDA Imagery Download & Site Coverage Optimization: `Planet`
This notebook authenticates with NASA CSDA credentials, loads STAC items alongside search site geometries, filters for 0% cloud cover, optimizes image selection to ensure maximum coverage per search site, downloads imagery assets, and visualizes the results.


| Product Type | Processing Level | Planet API Item Type | Planet API Asset Type Naming | Radiometry & Calibration | Band Configurations |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Basic Scene** | **Level 1B** | `PSScene` | • `basic_analytic_4b`<br>• `basic_analytic_8b` | **Top of Atmosphere (TOA)** radiance. | • 4-band (B, G, R, NIR)<br>• 8-band SuperDove |
| **Ortho Scene (TOA)** | **Level 3B** | `PSScene` | • `ortho_analytic_4b`<br>• `ortho_analytic_8b` | **Top of Atmosphere (TOA)** radiance. | • 4-band (B, G, R, NIR)<br>• 8-band SuperDove |
| **Ortho Scene (SR)** | **Level 3B** | `PSScene` | • `ortho_analytic_4b_sr`<br>• `ortho_analytic_8b_sr` | **Surface Reflectance (SR)** (Atmospherically corrected). | • 4-band (B, G, R, NIR)<br>• 8-band SuperDove |
| **Visual Scene** | **Level 3B** | `PSScene` | • `ortho_visual` | **Color-corrected** and enhanced for human viewing. | • 3-band Visual (RGB) |
| **Ortho Tile** | **Level 3B** | `ortho_tile` | • `ortho_analytic`<br>• `ortho_analytic_sr` | **TOA Radiance** or **Surface Reflectance (SR)**. | • 4-band or 5-band (legacy grid) |


In [1]:
import os
from pathlib import Path

# ==============================================================================
# INPUT CONFIGURATION
# ==============================================================================

# Credential File Path
CREDENTIALS_FILE = Path("/home/pmontesa/code/credentials.ini").expanduser()

YEAR = 'allyears' #2024

# Input STAC Items & GeoPackage File Paths
STAC_ITEMS_JSON = f"/home/pmontesa/code/csda_summaries/notebooks/stac_items_RadCalNet_planet_{YEAR}.json"
STAC_GPKG =       f"/home/pmontesa/code/csda_summaries/notebooks/stac_RadCalNet_planet_{YEAR}.gpkg"
SITES_GPKG =      f"/home/pmontesa/code/csda_summaries/sites/eval_sites_aoi.geojson"  # Path to your search sites shapefile/gpkg

# Output Directory for Downloaded Assets
OUTPUT_DIR = Path("/explore/nobackup/projects/CSDA_eval/csda_download/radcalnet_planet")

# Filtering & Optimization Settings
MIN_DATE = '2024-01-01'
MAX_CLOUD_COVER = 100.0  # Max cloud cover percentage allowed (0 to 100)
SELECT_TOP_PER_SITE = 10  # Minimum number of best-fitting images to select per site
ASSET_KEY_TO_DOWNLOAD = 'basic_analytic'  # Asset key in STAC Item (e.g., 'data', 'analytic', 'ortho')
DOWNLOAD_XML_ONLY = True  # Set to True to download XML metadata first before full raster
SITE_ID_COL = 'Site Name' # 'site_id'

# Create output folder if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
STAC_GPKG

'/home/pmontesa/code/csda_summaries/notebooks/stac_RadCalNet_planet_allyears.gpkg'

In [3]:
STAC_ITEMS_JSON

'/home/pmontesa/code/csda_summaries/notebooks/stac_items_RadCalNet_planet_allyears.json'

In [4]:
import configparser
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.plot import show
from matplotlib_scalebar.scalebar import ScaleBar
from pystac import ItemCollection
from httpx import BasicAuth
from csda_client import CsdaClient
import tabulate
import humanize

Matplotlib is building the font cache; this may take a moment.


## 1. NASA CSDA Authentication & Quota Verification

In [5]:
def load_earthdata_creds(path):
    config = configparser.ConfigParser()
    config.read(Path(path).expanduser())
    return config['EarthData']['username'], config['EarthData']['password']

username, password = load_earthdata_creds(CREDENTIALS_FILE)
client = CsdaClient.open(BasicAuth(username=username, password=password))

print(f"Auth Status: {client.verify()}")

# Display user quota profile
profile = client.profile(username)
rows = [[v.vendor, v.quota, v.quota_unit] for v in profile.vendors]
print("\nUser Quotas:")
print(tabulate.tabulate(rows, headers=["Vendor", "Quota", "Quota Unit"], tablefmt="simple"))

Auth Status: Hello montesano, you have a valid token!

User Quotas:
Vendor                                    Quota  Quota Unit
--------------------------------  -------------  ------------------
Airbus U.S.- Optical              1000000000000  QuotaUnit.filesize
Planet                                  5000000  QuotaUnit.area
Satellogic                        1000000000000  QuotaUnit.filesize
Vantor IKONOS (NASA only)                    -1  QuotaUnit.area
Vantor - WorldView 4 (NASA only)             -1  QuotaUnit.filesize
Vantor Legion                     1000000000000  QuotaUnit.filesize
Vantor (NASA only)                           -1  QuotaUnit.area


## 2. Load Data & Optimize Image Selection per Site
We filter by `cloud_cover == MAX_CLOUD_COVER` and calculate spatial intersections with search sites to pick the single best image (maximizing covered site area) for each site.

In [6]:
# 1. Load GeoDataFrames
stac_gdf = gpd.read_file(STAC_GPKG)

# 1. Convert the object column to datetime64
stac_gdf['datetime'] = pd.to_datetime(stac_gdf['datetime'], format='mixed')
# 2. Filter using a UTC-aligned Timestamp (Fixes the TypeError)
stac_gdf = stac_gdf[stac_gdf['datetime'] > pd.Timestamp(MIN_DATE, tz='UTC')]

sites_gdf = gpd.read_file(SITES_GPKG)
sites_gdf = sites_gdf[sites_gdf.Source == 'RadCalNet']

# Ensure CRS match
if stac_gdf.crs != sites_gdf.crs:
    sites_gdf = sites_gdf.to_crs(stac_gdf.crs)

# 2. Filter candidate imagery by Cloud Cover AND STAC Item Type == PSScene
cloud_mask = stac_gdf['pl:cloud_percent'] <= MAX_CLOUD_COVER

# Check ID prefix for 'PSScene-' (excluding PSScene4Band if desired)
is_psscene = stac_gdf['id'].str.startswith('PSScene')

# Combine filters
filtered_stac_gdf = stac_gdf[cloud_mask & is_psscene].copy()

# 3. Calculate Intersection Area with Sites to optimize coverage
equal_area_crs = "EPSG:3338"  # Alaska Albers Equal Area
stac_proj = filtered_stac_gdf.to_crs(equal_area_crs)
sites_proj = sites_gdf.to_crs(equal_area_crs)

#if 'site_id' not in sites_proj.columns:
sites_proj['site_id'] = sites_proj.index

# Spatial overlay intersection
intersection = gpd.overlay(sites_proj, stac_proj[['id', 'geometry']], how='intersection')
intersection['intersect_area_m2'] = intersection.geometry.area

# Rank images for each site by maximum intersecting coverage area
best_matches = (
    intersection.sort_values(by=['Site Name', 'intersect_area_m2'], ascending=[True, False])
    .groupby('Site Name')
    .head(SELECT_TOP_PER_SITE)
)

selected_item_ids = set(best_matches['id'].unique())

# 4. Load STAC ItemCollection and filter with deduplication
ic = ItemCollection.from_file(STAC_ITEMS_JSON)

seen_ids = set()
selected_stac_items = []

for item in ic.items:
    if item.id in selected_item_ids and item.id not in seen_ids:
        selected_stac_items.append(item)
        seen_ids.add(item.id)

print(f"Total Cloud-Free Candidate Images since {MIN_DATE}: {len(filtered_stac_gdf)}")
print(f"Optimized Selected Unique STAC Items for {len(sites_gdf)} Sites: {len(selected_stac_items)}")

Total Cloud-Free Candidate Images since 2024-01-01: 2125
Optimized Selected Unique STAC Items for 6 Sites: 50


### Check the types of data available for download

In [7]:
item = ic[-1]
rows = []
for key, asset in item.assets.items():
    if file_size := asset.ext.file.size:
        humanized_file_size = humanize.naturalsize(file_size)
    else:
        humanized_file_size = None
    if roles := asset.roles:
        humanized_roles = humanize.natural_list(roles)
    else:
        humanized_roles = roles
    rows.append([key, humanized_roles, asset.media_type, humanized_file_size])
tabulate.tabulate(
    rows, headers=["Key", "Roles", "Type", "File size", "Roles"], tablefmt="html"
)

Key,Roles,Type,File size
thumbnail,thumbnail,,
basic_udm2,data,image/tiff,3.1 MB
ortho_udm2,data,image/tiff,5.5 MB
ortho_visual,data,image/tiff,156.2 MB
json_metadata,metadata,application/json,970 Bytes
basic_analytic_8b,data,image/tiff,711.0 MB
ortho_analytic_8b,data,image/tiff,1.2 GB
ortho_analytic_8b_sr,data,image/tiff,1.0 GB
basic_analytic_4b_rpc,data,text/plain,3.3 kB
basic_analytic_8b_xml,metadata,text/xml,12.2 kB


In [8]:
# Standardize both sets to native Python integers
all_site_ids = set(int(x) for x in sites_proj['site_id']) if 'site_id' in sites_proj.columns else set(int(x) for x in sites_proj.index)
covered_site_ids = set(int(x) for x in best_matches['site_id'].unique())

# Calculate missing sites safely
missing_site_ids = sorted(list(all_site_ids - covered_site_ids))
covered_site_ids_sorted = sorted(list(covered_site_ids))

print(f"Covered Sites ({len(covered_site_ids_sorted)}): {covered_site_ids_sorted}")
print(f"Missing Sites ({len(missing_site_ids)}): {missing_site_ids}")

Covered Sites (5): [10, 31, 47, 52, 60]
Missing Sites (1): [63]


In [9]:
# 1. Group the best matches by STAC Image ID
image_coverage = (
    best_matches.groupby('id')
    .agg(
        site_count=('site_id', 'count'),
        covered_sites=('site_id', lambda x: ', '.join(map(str, sorted(x.unique()))))
    )
    .reset_index()
    .rename(columns={'id': 'stac_item_id'})
)

# 2. Display the summary table
print(f"Summary: {len(image_coverage)} images cover {best_matches['site_id'].nunique()} unique sites\n")
display(image_coverage)

Summary: 50 images cover 5 unique sites



,stac_item_id,site_count,covered_sites
0,PSScene-20240830_172737_13_24d1,1,60
1,PSScene-20240831_172620_51_24bd,1,60
2,PSScene-20240902_172718_39_24f9,1,60
3,PSScene-20240904_173040_41_24d3,1,60
4,PSScene-20240906_172603_19_24ee,1,60
5,PSScene-20240909_164319_18_24b4,1,60
6,PSScene-20240911_173055_98_24f9,1,60
7,PSScene-20240912_173023_09_24dd,1,60
8,PSScene-20240914_172926_41_24bd,1,60
9,PSScene-20240920_173243_97_251a,1,60


## 3. Download Data
Download selected assets or metadata XMLs using the `CsdaClient`.

In [10]:
# Priority order for raster downloads

PREFERRED_RASTER_TYPES = [
    ASSET_KEY_TO_DOWNLOAD,
    # Modern PSScene (8-band & 4-band)
    "basic_analytic_8b",
    # "ortho_analytic_8b_sr",
    # "ortho_analytic_4b_sr",
    # "ortho_analytic_8b",
    # "ortho_analytic_4b",
    # # Legacy PSScene4Band & PSScene3Band
    # "analytic_sr",
    # "analytic",
    # # Visual assets (fallbacks)
    # "ortho_visual",
    # "visual"
]

# Priority order for metadata downloads
PREFERRED_XML_TYPES = [
    "basic_analytic_8b_xml",
    # "ortho_analytic_8b_sr_xml",
    # "ortho_analytic_4b_sr_xml",
    
    # # Modern PSScene XML types
    # "ortho_analytic_8b_xml",
    # "ortho_analytic_4b_xml",

    # "basic_analytic_4b_xml",
    # # Legacy PSScene4Band & PSScene3Band XML types
    # "analytic_xml",
    # "analytic_sr_xml",
    
    # # Generic / Tile XML fallbacks
    # "xml"
]

# List to store information dictionaries for each downloaded image
download_info_list = []
processed_item_ids = set()

for item in selected_stac_items:
    print(f"\nProcessing STAC Item: {item.id}")
    
    # Extract collection name (fallback if collection is not populated)
    collection_name = item.collection_id or item.collection or "csda_data"
    
    # 1. Download XML/Metadata with custom filename
    xml_type = next((k for k in PREFERRED_XML_TYPES if k in item.assets), None)
    if (DOWNLOAD_XML_ONLY or xml_type) and xml_type:
        try:
            ext = ".json" if "json" in xml_type else ".xml"
            xml_filename = f"{collection_name}_{item.id}_{xml_type}{ext}"
            xml_target_path = OUTPUT_DIR / xml_filename
            
            if xml_target_path.exists():
                print(f"  Metadata already downloaded: {xml_target_path}")
            else:
                print(f"  Downloading metadata ({xml_type})...")
                client.download_item(item, xml_type, str(xml_target_path))
                print(f"  Saved metadata to: {xml_target_path}")
        except Exception as e:
            print(f"  Could not download metadata for {item.id}: {e}")

    # 2. Download Primary Raster Asset
    if not DOWNLOAD_XML_ONLY:
        raster_type = next((k for k in PREFERRED_RASTER_TYPES if k in item.assets), None)
        
        if raster_type:
            raster_filename = f"{collection_name}_{item.id}_{raster_type}.tif"
            raster_target_path = OUTPUT_DIR / raster_filename
            
            stac_gdf_sub = stac_gdf[stac_gdf['id'] == item.id].copy()
            
            info_dict = {
                "local_path": str(raster_target_path),
                "stac_gdf": stac_gdf_sub,
                "sites_gdf": sites_gdf,
                "item_id": item.id
            }
            
            if raster_target_path.exists() and raster_target_path.stat().st_size > 0:
                print(f"  File already exists on disk. Skipping download: {raster_target_path}")
                download_info_list.append(info_dict)
                processed_item_ids.add(item.id)
            else:
                print(f"  Downloading raster asset ('{raster_type}')...")
                try:
                    client.download_item(item, raster_type, str(raster_target_path))
                    print(f"  Saved to: {raster_target_path}")
                    download_info_list.append(info_dict)
                    processed_item_ids.add(item.id)
                except Exception as e:
                    print(f"  Download failed for {item.id} ({raster_type}): {e}")


Processing STAC Item: PSScene-20241231_034629_27_24ae
  Could not download metadata for PSScene-20241231_034629_27_24ae: Client error '403 Forbidden' for url 'https://csdap.earthdata.nasa.gov/api/v2/download/planet/PSScene-20241231_034629_27_24ae/basic_analytic_8b_xml'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

Processing STAC Item: PSScene-20241225_034547_51_24ed
  Could not download metadata for PSScene-20241225_034547_51_24ed: Client error '403 Forbidden' for url 'https://csdap.earthdata.nasa.gov/api/v2/download/planet/PSScene-20241225_034547_51_24ed/basic_analytic_8b_xml'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

Processing STAC Item: PSScene-20241224_034511_89_24de
  Could not download metadata for PSScene-20241224_034511_89_24de: Client error '403 Forbidden' for url 'https://csdap.earthdata.nasa.gov/api/v2/download/planet/PSScene-20241224_034511_89_24de/basic_analytic_8b_xml'
For more 

In [11]:
len(download_info_list)

0

In [148]:
import pickle
from pathlib import Path

# Path to output directory
save_path = OUTPUT_DIR / f"download_info_list_{YEAR}.pkl"

with open(save_path, "wb") as f:
    pickle.dump(download_info_list, f)

print(f"Successfully saved download_info_list ({len(download_info_list)} items) to:")
print(save_path)

Successfully saved download_info_list (0 items) to:
/explore/nobackup/projects/CSDA_eval/csda_download/radcalnet_planet/download_info_list_allyears.pkl
